# Baseline: что умеет готовый Donut до дообучения

Этот ноутбук показыает отправную точку перед дообучением. Без честно измеренного baseline мы не сможем потом доказать, что дообучение дало результат.

Donut это модель, которая принимает фотографию документа и сразу выдаёт его содержимое структурой, без отдельного распознавания текста. Мы используем версию, дообученную авторами на датасете CORD. На CORD размечены товарные позиции и итоговые суммы, поэтому готовая модель должна хорошо извлекать именно их.

Но в нашем проекте нужны ещё и реквизиты чека: магазин, дата, адрес. Этих полей в CORD не было, и готовая модель их извлекать не училась. Чтобы понять, что именно стоит дообучать, мы измеряем baseline на двух датасетах сразу и смотрим на контраст.

На CORD проверяем, насколько хорошо модель извлекает то, на чём обучена, прежде всего итоговую сумму.

На SROIE, где размечены реквизиты, проверяем магазин, дату, адрес и сумму. Ожидаем, что по реквизитам модель будет слаба, потому что не училась этим полям. Именно этот разрыв мы и закроем дообучением в следующем ноутбуке.

## Запуск в Google Colab

Ноутбук рассчитан на запуск в Colab без ручных скачиваний:
код клонируется из GitHub, картинки SROIE (`sroie_donut`) скачиваются zip-архивом
с Google Drive, модель Donut и CORD приходят с HuggingFace Hub.


In [ ]:
# Настройка окружения для Colab
import os
import sys
import subprocess
from pathlib import Path
import zipfile
import gdown


IN_COLAB = "google.colab" in sys.modules
# Публичный git-репозиторий с кодом проекта
REPO_URL = "https://github.com/ScarletFlame611/Receipt-AI.git"
# id zip-архива
SROIE_DONUT_GDRIVE_ID = "10nAlGwwvjI0TLhl0JXusYug_KuqZzvfX"


def _find_project_root():
    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / "src").is_dir():
            return cand
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "src").is_dir():
            return sub
    return None


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "datasets", "transformers", "seaborn", "gdown", "pillow-heif", "zss"],
        check=False,
    )
    if _find_project_root() is None and "USER/" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL], check=True)

for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]
root = _find_project_root()
if root is None:
    raise RuntimeError(
        "Не найден код проекта"
    )
os.chdir(root)
sys.path.insert(0, str(root))
print("Colab:", IN_COLAB, "| корень проекта:", root)


def ensure_sroie_donut(root):
    target = Path(root) / "data" / "processed" / "sroie_donut"
    if (target / "train" / "metadata.jsonl").exists():
        return target

    target.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target.parent / "sroie_donut.zip"
    print("sroie_donut не найден локально — качаю с Google Drive...")
    gdown.download(id=SROIE_DONUT_GDRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target.parent)
    zip_path.unlink()
    print("Готово:", target)
    return target

## Настройка окружения

Проверяем GPU, ставим библиотеки. Donut работает через transformers, метрика древесного расстояния через библиотеку zss.

In [11]:
import torch
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))

GPU доступен: True
Устройство: Tesla T4


In [2]:
!pip install transformers datasets zss -q

  Preparing metadata (setup.py) ... done


In [12]:
from pathlib import Path

ensure_sroie_donut(root)
data_root = root / "data" / "processed" / "sroie_donut"
print("датасет на месте:", data_root.exists())
for split in ["train", "validation", "test"]:
    meta = data_root / split / "metadata.jsonl"
    if meta.exists():
        n = len(meta.read_text(encoding="utf-8").strip().splitlines())
        print(f"{split}: {n} чеков")
    else:
        print(f"{split}: нет metadata.jsonl")

Mounted at /content/drive
датасет на месте: True
train: 500 чеков
validation: 62 чеков
test: 64 чеков


In [13]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

model_name = "naver-clova-ix/donut-base-finetuned-cord-v2"
processor = DonutProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Donut загружен на", device)

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Donut загружен на cuda


## Baseline на CORD

Начинаем с CORD, потому что именно на нём готовая модель должна показать себя хорошо. Если подтвердится, что она уже отлично извлекает сумму, это будет означать, что дообучать её на CORD незачем, и мы используем её как готовый компонент для товарных позиций.

Сначала создадим функцию, которая прогоняет Donut на одном изображении. Она готовит картинку, задаёт модели стартовую подсказку задачи, генерирует последовательность токенов и преобразует её обратно в структуру-словарь. Эту же функцию используем дальше и для SROIE.

In [14]:
def run_donut(image):
    pixel_values = processor(image.convert("RGB"), return_tensors="pt").pixel_values.to(device)
    task_prompt = "<s_cord-v2>"
    decoder_input_ids = processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(device)
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=768,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )
    seq = processor.batch_decode(outputs.sequences)[0]
    seq = seq.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    return processor.token2json(seq)

Загружаем тестовую часть CORD и определяем, как сравнивать суммы. Сумму проверяем в двух режимах. Строгий режим требует совпадения строки символ в символ. Нормализованный сначала приводит обе суммы к числу, отбрасывая разделители, и сравнивает уже числа. Разница между этими двумя цифрами покажет, сколько модель теряет на форматировании, а не на реальных ошибках.

In [15]:
from datasets import load_dataset
import json, re

cord_test = load_dataset("naver-clova-ix/cord-v2")["test"]

def parse_amount_loose(raw):
    if raw is None:
        return None
    digits = re.sub(r"[^\d]", "", str(raw))
    return int(digits) if digits else None

def get_total(parse):
    total = parse.get("total")
    if isinstance(total, dict):
        return total.get("total_price")
    return None

print("тестовых чеков CORD:", len(cord_test))

тестовых чеков CORD: 100


Помимо точности суммы посчитаем древесное расстояние, стандартную метрику качества для Donut. Она сравнивает всю предсказанную структуру с эталонной целиком, а не одно поле. Структуру представляем деревом и считаем, сколько правок нужно, чтобы превратить предсказание в эталон, нормируя на размер эталона. Ноль означает идеальное совпадение, чем больше значение, тем сильнее расхождение.

In [16]:
import zss

class _Node:
    def __init__(self, label, children=None):
        self.label = label
        self.children = children or []

def _dict_to_tree(obj, label="root"):
    node = _Node(label)
    if isinstance(obj, dict):
        for key in sorted(obj.keys()):
            node.children.append(_dict_to_tree(obj[key], key))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            node.children.append(_dict_to_tree(item, f"[{i}]"))
    else:
        node.children.append(_Node(str(obj)))
    return node

def _tree_size(node):
    return 1 + sum(_tree_size(c) for c in node.children)

def tree_edit_distance(pred, ref):
    pt = _dict_to_tree(pred if isinstance(pred, dict) else {})
    rt = _dict_to_tree(ref)
    dist = zss.simple_distance(pt, rt, lambda n: n.children, lambda n: n.label)
    size = _tree_size(rt)
    return dist / size if size else 0.0

Прогоняем модель по всем тестовым чекам CORD. Для каждого сохраняем предсказание и эталон, чтобы потом посчитать метрики.

In [17]:
from tqdm import tqdm

cord_preds, cord_refs = [], []
for example in tqdm(cord_test, desc="CORD"):
    cord_preds.append(run_donut(example["image"]))
    cord_refs.append(json.loads(example["ground_truth"])["gt_parse"])

print("обработано:", len(cord_preds))

CORD: 100%|██████████| 100/100 [01:18<00:00,  1.28it/s]

обработано: 100


In [18]:
import numpy as np

strict_hits = norm_hits = both_have = 0
for pred, ref in zip(cord_preds, cord_refs):
    ref_total = get_total(ref)
    if ref_total is None:
        continue
    both_have += 1
    pred_total = get_total(pred) if isinstance(pred, dict) else None
    if pred_total is not None:
        if str(pred_total).strip() == str(ref_total).strip():
            strict_hits += 1
        if parse_amount_loose(pred_total) == parse_amount_loose(ref_total):
            norm_hits += 1

cord_ted = [tree_edit_distance(p, r) for p, r in zip(cord_preds, cord_refs)]
cord_results = {
    "сумма строгая": strict_hits / both_have,
    "сумма нормализованная": norm_hits / both_have,
    "TED среднее": float(np.mean(cord_ted)),
    "TED медиана": float(np.median(cord_ted)),
}
print("Baseline на CORD (чеков с суммой:", both_have, ")")
for k, v in cord_results.items():
    print(f"{k}: {round(v, 3)}")

Baseline на CORD (чеков с суммой: 95 )
сумма строгая: 0.916
сумма нормализованная: 0.979
TED среднее: 0.532
TED медиана: 0.067


### Что показывает baseline на CORD

Готовая модель на CORD уже работает хорошо, как и ожидалось. Итоговую сумму она извлекает правильно почти всегда: при сравнении по числу точность составляет 0.979, то есть из 95 чеков с размеченной суммой модель верно определила её почти во всех. При строгом посимвольном сравнении цифра ниже, 0.916, и эта разница примерно в шесть процентов это ровно потеря на форматировании т.е. сумма найдена верно, но записана в чуть ином виде, например с другими разделителями. Само по себе это не ошибка модели, а вопрос приведения к единому формату, чем займётся модуль нормализации.

Древесное расстояние подтверждает картину. Медианное значение очень низкое, 0.067, то есть для половины чеков структура практически идеальна. Среднее заметно выше, 0.532, и этот разрыв между медианой и средним означает, что есть небольшое число чеков, на которых структура разъезжается сильно, и они тянут среднее вверх. Но таких меньшинство.

Дообучать модель на CORD не имеет смысла, расти почти некуда. Поэтому в нашем проекте мы используем готовую модель как есть для извлечения товарных позиций и суммы. А теперь посмотрим на то, чему она не училась.

## Baseline на SROIE

Переходим к реквизитам. На датасете SROIE размечены магазин, дата, адрес и итоговая сумма. Прогоняем по его тестовой части ту же самую готовую модель и смотрим, насколько она справляется с полями, которых не было в её обучении.

Прежде чем считать метрики по всем чекам, посмотрим на один пример.

In [19]:
from PIL import Image

sroie_test_dir = data_root / "test"
sroie_test = [json.loads(l) for l in
              (sroie_test_dir / "metadata.jsonl").read_text(encoding="utf-8").strip().splitlines()]
ex = sroie_test[0]
img = Image.open(sroie_test_dir / ex["file_name"])
ref = json.loads(ex["ground_truth"])["gt_parse"]
pred = run_donut(img)
print("эталон (что нужно извлечь):")
print(json.dumps(ref, ensure_ascii=False, indent=2))
print("\nпредсказание готовой модели:")
print(json.dumps(pred, ensure_ascii=False, indent=2))

эталон (что нужно извлечь):
{
  "company": "GARDENIA BAKERIES (KL) SDN BHD",
  "date": "29/10/2017",
  "address": "LOT 3, JALAN PELABUR 23/1, 40300 SHAH ALAM, SELANGOR.",
  "total": "58.86"
}

предсказание готовой модели:
{
  "menu": [
    {
      "nm": "GARDENIA BAKERIES (KL) SDN BHD (139386 X)",
      "unitprice": "1399040",
      "cnt": {
        "unitprice": "1399040"
      },
      "price": "(139386 X)"
    },
    {
      "nm": "GST II): Fax:03-",
      "unitprice": "55423228",
      "discountprice": "NOTE",
      "price": "55423213"
    }
  ],
  "sub_total": {
    "subtotal_price": "1399040",
    "tax_price": "55423213"
  },
  "total": {
    "total_price": "29/10/2017",
    "cashprice": "Cash Inv No.: 7029F711",
    "changeprice": "29/10/2017",
    "creditcardprice": [
      {
        "nm": "GROUND FLOOR, NO. 4 & 6,",
        "unitprice": "6,",
        "cnt": [
          {
            "nm": "SUBANG JAYA, SELANGOR",
            "num": "VEOS: Ridzuan (11900)",
            "price": 

Один пример уже показывает что готовая модель пытается описать чек в терминах меню и цен и не выделяет реквизиты как отдельные поля. Теперь измерим это по всем 64 тестовым чекам.

Сравнение полей строим так же, как для суммы. Текстовые поля, магазин и адрес, сравниваем после простой нормализации: приводим к верхнему регистру и убираем лишние пробелы, чтобы не штрафовать за мелкие расхождения в написании. Дату сравниваем как строку после очистки пробелов. Сумму, как и раньше, по числу. Из предсказания модели достаём поля верхнего уровня company, date, address, total, если они там вообще есть.

In [20]:
def norm_text(s):
    if s is None:
        return None
    return " ".join(str(s).upper().split())

def get_field(parse, key):
    if isinstance(parse, dict):
        val = parse.get(key)
        if isinstance(val, (str, int, float)):
            return val
    return None

def field_match(pred_val, ref_val, kind):
    if ref_val is None:
        return None
    if pred_val is None:
        return False
    if kind == "amount":
        return parse_amount_loose(pred_val) == parse_amount_loose(ref_val)
    return norm_text(pred_val) == norm_text(ref_val)

Прогоняем готовую модель по всем тестовым чекам SROIE и для каждого поля считаем долю верных извлечений.

In [21]:
fields = ["company", "date", "address", "total"]
kinds = {"company": "text", "date": "text", "address": "text", "total": "amount"}

hits = {f: 0 for f in fields}
totals = {f: 0 for f in fields}
sroie_preds, sroie_refs = [], []
for ex in tqdm(sroie_test, desc="SROIE"):
    img = Image.open(sroie_test_dir / ex["file_name"])
    ref = json.loads(ex["ground_truth"])["gt_parse"]
    pred = run_donut(img)
    sroie_preds.append(pred)
    sroie_refs.append(ref)
    for f in fields:
        m = field_match(get_field(pred, f), ref.get(f), kinds[f])
        if m is not None:
            totals[f] += 1
            if m:
                hits[f] += 1
print("\nBaseline на SROIE:")
for f in fields:
    acc = hits[f] / totals[f] if totals[f] else 0.0
    print(f"{f}: {round(acc, 3)}  ({hits[f]} из {totals[f]})")

SROIE: 100%|██████████| 64/64 [03:08<00:00,  2.95s/it]


Baseline на SROIE (готовая модель, без дообучения):
company: 0.0  (0 из 64)
date: 0.0  (0 из 64)
address: 0.0  (0 из 63)
total: 0.0  (0 из 64)


### Что показывает baseline на SROIE

Ииии, мы видим ноль по всем четырём полям. Готовая модель не извлекла правильно ни одного магазина, ни одной даты, адреса или суммы в том виде, в каком они нужны.

Важно понимать, что это не значит, будто модель совсем слепа к тексту чека. Как мы видели на разобранном примере, она читает символы и даже улавливает и название магазина, и сумму. Проблема в другом: она раскладывает прочитанное по структуре, которой училась на CORD, то есть по меню, ценам и подытогам. Реквизитов как отдельных полей верхнего уровня в её картине мира не существует, поэтому в нужном месте их нет, и для нашей задачи это равносильно нулю.

Именно этот ноль и задаёт точку отсчёта: сейчас модель не умеет извлекать реквизиты вовсе. После дообучения на SROIE мы прогоним ту же проверку на тех же тестовых чеках и сравним. Любой заметный результат будет наглядным ростом, а мы рассчитываем на высокую точность, потому что задача для модели посильная: текст она уже читает, нужно лишь научить её складывать его в правильные поля.

## Итог: что мы измерили и что делаем дальше

На CORD готовая модель сильна. Сумму извлекает почти безупречно, структура для большинства чеков близка к идеалу. Дообучать её на CORD незачем, поэтому в проекте мы используем её как готовый компонент для товарных позиций и суммы.

На SROIE та же модель показала ноль по всем реквизитам. Магазин, дату и адрес она не извлекает, потому что им не училась. Это и есть тот разрыв, который нужно закрыть.

Товарные позиции берём из готовой модели на CORD. А для реквизитов дообучаем Donut на SROIE и доводим точность по магазину, дате, адресу и сумме с нуля до рабочего уровня. Этим займёмся в следующем ноутбуке, где будем обучать модель и затем вернёмся к этим же тестовым чекам, чтобы честно измерить рост.

Зафиксированные стартовые цифры: на CORD сумма около 0.98, на SROIE все реквизиты по нулю. С ними и будем сравнивать.